# Deep learning: frozen embeddings, fine-tuned DistilBERT, zero-shot LLM (notebook 03)

Session 7 order: first freeze a pretrained encoder and train only a small head on its
embeddings (cheap, cached), then fine-tune the whole encoder (expensive). Last, a zero-shot
language model (Session 8) answers "is this ad fraudulent?" with no training at all. Each model is scored
once on test per (model, seed) on the same five splits as notebook 02; comparing and plotting
the saved results is notebook 04. Heavy cells need a GPU and the `torch`, `transformers` and
`sentence-transformers` packages (plus `captum` for the explanation); the run cells are skipped unless `ON_GPU_SERVER = True`
(set it on the server only).

Run time: one DistilBERT run took about 7 minutes on one GPU at batch size 128 (measured, seed 0,
5 epochs); the embedding heads take seconds once the embeddings are cached.

In [ ]:
import sys, os; os.chdir(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else None; sys.path.insert(0, "."); import warnings
import importlib.util

from modules import data_helper as dh
from modules import features as ft
from modules import deep_runs as dr

DATA = "../data/fake_job_postings.csv"

In [ ]:
SEED = 0
SEEDS = [0, 1, 2, 3, 4]
BERT_SEEDS = SEEDS  # seeds with existing result files are skipped
OUT_DEEP = "outputs/deep"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BERT_MODEL = "distilbert-base-uncased"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS, BATCH_SIZE, MAX_LEN, LR = 5, 128, 256, 2e-5
LOG_EVERY = 10  # print the loss every this many steps
DEVICE = "cuda:1"  # which GPU to use; the server is shared, so pick an emptier one (nvidia-smi), e.g. "cuda:1"

# SAFETY: the embedding and fine-tuning cells are heavy GPU work. They run only when this flag is
# True, and it must stay False on the laptop. Set it to True on the GPU server, nowhere else.
ON_GPU_SERVER = False

# Libraries present? (a missing one would make the heavy cells fail, not skip)
HAVE_TORCH = all(importlib.util.find_spec(m)
                 for m in ("torch", "transformers", "sentence_transformers", "captum"))
HAVE_EMBEDDINGS = os.path.exists("features/" + EMBED_MODEL.replace("/", "_") + "_emb.npy")

# Heavy cells need the explicit opt-in AND the libraries; the sklearn heads are light (CPU) and
# only need the cached embeddings.
RUN_EMBED = RUN_BERT = RUN_LLM = ON_GPU_SERVER and HAVE_TORCH
RUN_HEADS = RUN_EMBED or HAVE_EMBEDDINGS
print({"ON_GPU_SERVER": ON_GPU_SERVER, "torch stack": HAVE_TORCH, "cached embeddings": HAVE_EMBEDDINGS})

dh.seed_everything(SEED)

## Data and splits

The same cleaned frame and the same saved splits as the classical notebook, so every model
is scored on identical rows.

In [ ]:
df = ft.add_engineered_features(dh.load_processed())
print("rows:", len(df), "| fraud:", int(df["fraudulent"].sum()))

## Step 1: frozen sentence embeddings

`all-MiniLM-L6-v2` turns each ad (the joined `text` column) into one 384-dimensional vector.
Nothing is trained here. The vectors are cached in `features/` keyed by `job_id`, so this
cell is a no-op the second time.

In [ ]:
if RUN_EMBED:
    # Encode every ad once; the result is cached in features/.
    dr.run_embed(df, DEVICE, EMBED_MODEL)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + sentence-transformers")

## Step 2: logistic regression on the frozen embeddings

A logistic regression (C picked on validation PR-AUC) trained on the cached vectors with CPU only.
It is the same classifier as the TF-IDF logistic regression in notebook 02, so the pair compares
the text representation alone. Run for all five seeds; the threshold is picked on validation and
the test rows are scored once.

In [ ]:
if RUN_HEADS:
    dr.run_heads(df, SEEDS, OUT_DEEP, EMBED_MODEL)
else:
    print("skipped: no cached embeddings here")

## Step 3: fine-tune DistilBERT

The whole encoder is trained with a linear classification layer on top: AdamW, learning rate
2e-5 with warm-up and linear decay, square-root class weights in the loss, bf16 autocast on
CUDA, and early stopping on validation PR-AUC (the best epoch's weights are restored). Texts
are truncated to 256 tokens. `BERT_SEEDS` lists the seeds to fine-tune; finished ones are skipped.

Model explanation: right after seed 0 is trained, Integrated Gradients scores how much each token
pushed the model towards fraud, for 10 seed-0 test ads (4 caught frauds, 3 missed frauds, 3 false
alarms). They are saved next to the results and shown in notebook 04.

In [ ]:
if RUN_BERT:
    dr.run_bert(df, BERT_SEEDS, OUT_DEEP, DEVICE, BERT_MODEL, epochs=EPOCHS,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, lr=LR, log_every=LOG_EVERY)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers")

## Step 4: zero-shot language model

`Qwen2.5-1.5B-Instruct` reads each ad (cut to 512 tokens) in a short Yes/No prompt; the fraud
score is its probability of answering "Yes" (renormalised over Yes and No). Nothing is trained,
so every ad is scored once and the scores are cached in `features/`; each seed then only picks
its threshold on validation and scores its test rows, like every other model.

In [ ]:
if RUN_LLM:
    dr.run_zero_shot(df, SEEDS, OUT_DEEP, DEVICE, LLM_MODEL)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers")